In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./models/checkpoint-1875"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [10]:
prompt = """def add(a: int, b):"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=.2,
    do_sample=True,
    # top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

completion = tokenizer.decode(output_tokens[0], skip_special_tokens=False)
print(completion)

def add(a: int, b): int = a + b

def add(a: int, b: int): int = a + b

def add(a: int, b: int): int = a + b

def add(a: int, b: int): int = a + b

def add(a: int, b: int): int = a + b

def add(a: int, b: int): int = a + b

def add(


# Experimentation

In dieser Section schauen wir wie sich unterschiedliche parameter auf den output auswirken. Wir nehmen als Beispiel eine simple Add funktion die in Python generiert werden soll.

In [ ]:
# helper function
def code_complete(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_tokens = model.generate(
        **inputs,
        **kwargs
    )

    completion = tokenizer.decode(output_tokens[0], skip_special_tokens=False)
    print(completion)


In [12]:
code_complete(
    "def add(a: int, b: int):",
    max_new_tokens=100,
    temperature=.2,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int): int {
    return a + b
}

def add(a: int, b: int): int {
    return a + b
}

def add(a: int, b: int): int {
    return a + b
}

def add(a: int, b: int): int {
    return a + b
}

def add(a: int, b: int):


Das Modell scheint in Scala zu antworten und sich endlos zu wiederholen. Das könnte an der niedrigen temperatur liegen (`0.2`)

In [13]:
code_complete(
    "def add(a: int, b: int):",
    max_new_tokens=100,
    temperature=.9,  # higher temperature
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int): int = a + b
    print ("added %.2f to %.2f" % (a, b))
<|endoftext|>


Eine hohe Temperatur (`0.9`) bricht zwar den Loop, aber das Modell antwortet nichts brauchbares. Immernoch Scala.

### Top-k und Top-p


In [27]:
code_complete(
    "def add(a: int, b: int):",
    max_new_tokens=100,
    temperature=.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int): int = a + b

def main():
    import matplotlib.pyplot as plt
    plt.show()

def main():
    import matplotlib.pyplot as plt
    plt.show()

if __name__ == '__main__':
    main()

If you want to know what I did wrong, please let me know.

A:


In [31]:
code_complete(
    "def add(a: int, b: int):",
    max_new_tokens=100,
    temperature=.3,
    top_k=50,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int): int {
        return a + b
    }
    def add(a: int, b: int): int {
        return a + b
    }
    def add(a: int, b: int): int {
        return a + b
    }
    def add(a: int, b:


immernoch scala. 

### Besserer Prompt (Context Setting)

Das Modell hat bis jetzt immer nur im Scala Syntax `int = a + b` geantwortet. Ein versuch ist es durch einen Docstring die funktionsbeschreibung anzupassen.

In [25]:
python_prompt = '''def add(a: int, b: int):
    """Adds two numbers and returns the result"""'''

code_complete(
    python_prompt,
    max_new_tokens=50,
    temperature=.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int):
    """Adds two numbers and returns the result"""
    return a + b

def add(a: int, b: int):
    """Adds two numbers and returns the result"""
    return a + b

def add(a: int


hat funktioniert!

aber das modell widerholt sich immernoch

### Do Sample = False

In [33]:
# 3. Dein neuer Versuch
python_prompt = """def add(a: int, b: int):
    \"\"\"Adds two numbers in Python and returns the result.\"\"\""""

code_complete(
    python_prompt,
    max_new_tokens=100,  
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """


funktionert mit diesem Prompt auch. Wiederholt sich aber immernoch.

### Repetition Penalty

In [ ]:
code_complete(
    python_prompt,
    max_new_tokens=100,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.3,  
)

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b

def add(a: int, b: int):
    """


In [ ]:
code_complete(
    python_prompt,
    max_new_tokens=50,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=3.,  
)

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    return a + b
<|endoftext|>


bei hoch genuger Repetition_penalty kann es sein dass es funktioniert (der obere output hat ca 5 mal ausführen gebraucht). Ansonsten kommt immernoch halluzinierungen und random for loops etc.

In [ ]:
# nochmal geliche parameter:
code_complete(
    python_prompt,
    max_new_tokens=50,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=3.,  
)

def add(a: int, b: int):
    """Adds two numbers in Python and returns the result."""
    a = a + 1
    b = b + 2
    return (a, b)

def main():
    import sys
    from pytest.tasks import


gleicher parameter, aber mehr als die gefragte funktion